DAY 61

Part 1: What dtypes do you actually have?

In [1]:
import pandas as pd
import sys


sys.path.insert(0, '..')
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

In [2]:
# 2

report.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   transaction_id            200 non-null    str  
 1   lc_number                 198 non-null    str  
 2   lc_form                   200 non-null    str  
 3   available_with            200 non-null    str  
 4   issue_date                198 non-null    str  
 5   expiry_date               200 non-null    str  
 6   latest_shipment_date      200 non-null    str  
 7   applicant_name            199 non-null    str  
 8   applicant_address         198 non-null    str  
 9   applicant_city            200 non-null    str  
 10  applicant_postal_code     200 non-null    str  
 11  applicant_country         200 non-null    str  
 12  applicant_lei             199 non-null    str  
 13  applicant_account         200 non-null    str  
 14  beneficiary_name          200 non-null    str  
 15  

Part 2: Categorical — The Big Memory Win

In [3]:
# 3
#  Check current memory usage of one column
before = report['validation_status'].memory_usage(deep=True)

# Convert to category
report['validation_status'] = report['validation_status'].astype('category')

after = report['validation_status'].memory_usage(deep=True)

print(f"Before: {before:,} bytes")
print(f"After:  {after:,} bytes")
print(f"Saved:  {(1 - after/before) * 100:.1f}%")




Before: 2,988 bytes
After:  361 bytes
Saved:  87.9%


In [4]:
# 4
# Reset to original first
report['validation_status'] = report['validation_status'].astype('str')

# Measure full DataFrame
before_total = report.memory_usage(deep=True).sum()

# Convert several string columns that have few unique values
category_columns = ['validation_status', 'severity', 'currency', 'lc_form',
                     'available_with', 'confirmation_status', 'incoterm',
                     'partial_shipment', 'transhipment_allowed']

for col in category_columns:
    report[col] = report[col].astype('category')

after_total = report.memory_usage(deep=True).sum()

print(f"Before: {before_total:,} bytes")
print(f"After:  {after_total:,} bytes")
print(f"Saved:  {(1 - after_total/before_total) * 100:.1f}%")

Before: 187,403 bytes
After:  161,136 bytes
Saved:  14.0%


## The rule of thumb: if nunique() / len() < 0.5,  it's a candidate.

In [5]:
# 5
# Find good category candidates
str_cols = report.select_dtypes(include='object').columns

ratios = []
for col in str_cols:
    unique_ratio = report[col].nunique() / len(report)
    ratios.append({'column': col, 'unique_ratio': round(unique_ratio, 3), 'nunique': report[col].nunique()})

candidates = pd.DataFrame(ratios).sort_values('unique_ratio')
candidates.head(15)


/tmp/ipykernel_11828/2520061557.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = report.select_dtypes(include='object').columns


,column,unique_ratio,nunique
31,documents_required,0.005,1
33,payment_terms,0.015,3
34,charges,0.015,3
21,issuing_bank_country,0.030,6
16,beneficiary_country,0.035,7
9,applicant_country,0.040,8
32,additional_conditions,0.050,10
24,confirming_bank_name,0.055,11
25,confirming_bank_swift,0.055,11
19,issuing_bank_name,0.060,12


convert ALL the good candidates and measure:

In [6]:
# 6
# Find and convert all good candidates (ratio < 0.1)
good_candidates = candidates.query("unique_ratio < 0.1")['column'].tolist()
print(f"Converting {len(good_candidates)} columns to category") # 22

before = report.memory_usage(deep=True).sum()

for col in good_candidates:
    report[col] = report[col].astype('category')

after = report.memory_usage(deep=True).sum()
print(f"Before: {before:,} bytes")
print(f"After:  {after:,} bytes")
print(f"Saved:  {(1 - after/before) * 100:.1f}%")



Converting 22 columns to category
Before: 161,136 bytes
After:  93,359 bytes
Saved:  42.1%


## Part 3: Numeric Downcasting

In [7]:
#7
# Now the integer columns. int64 can hold values up to 9 quintillion. Your tolerance_percent maxes out at maybe 10. Massive overkill

# check the ranges first
int_cols = ['tolerance_percent', 'presentation_period_days', 'error_count']
for col in int_cols:
    print(f"{col}: min={report[col].min()}, max={report[col].max()}, dtype={report[col].dtype}")





tolerance_percent: min=0, max=10, dtype=int64
presentation_period_days: min=14, max=21, dtype=int64
error_count: min=0, max=6, dtype=int64


In [8]:
# 8
before = report.memory_usage(deep=True).sum()

for col in  int_cols:
    report[col] = pd.to_numeric(report[col], downcast='unsigned')
    print(f"{col} -> {report[col].dtype}")

after = report.memory_usage(deep=True).sum()
print(f"\nSaved: {before - after:,} bytes")

tolerance_percent -> uint8
presentation_period_days -> uint8
error_count -> uint8

Saved: 4,200 bytes


## Part 4: Datetime conversion

In [9]:
#9
date_cols = ['issue_date', 'expiry_date', 'latest_shipment_date']
for col in date_cols:
    report[col] = pd.to_datetime(report[col], errors='coerce')

report[date_cols].dtypes

# errors='coerce' is your defensive friend — bad date strings become NaT (Not a Time, the datetime version of NaN) instead of crashing

issue_date              datetime64[us]
expiry_date             datetime64[us]
latest_shipment_date    datetime64[us]
dtype: object

datetime64[us] — proper datetime objects now. Now you can do real date math:

In [10]:
# 10
# datetime64[us] — proper datetime objects now. Now you can do real date math:

# Compute LC validity period in days (the same calc your DateValidator does!)
report['validity_days'] = (report['expiry_date'] - report['issue_date']).dt.days

# Quick stats
report['validity_days'].describe()

count     196.000000
mean      171.994898
std       204.447937
min       -92.000000
25%       135.000000
50%       153.500000
75%       165.250000
max      2690.000000
Name: validity_days, dtype: float64

In [11]:
# 11
# Transactions with weird validity periods
weirdos = report.query("validity_days < 0 or validity_days > 540")
weirdos[['transaction_id', 'issue_date', 'expiry_date', 'validity_days', 'error_codes']]


,transaction_id,issue_date,expiry_date,validity_days,error_codes
3,LC-0OXFQ8,2026-01-10,2028-06-10,882.0,"LEI004, DATE007"
15,LC-6DPBHS,2019-01-15,2026-05-28,2690.0,DATE007
67,LC-5IGQPK,2026-01-10,2028-06-10,882.0,DATE007
106,LC-PUXQPH,2026-06-01,2026-03-01,-92.0,"LEI004, DATE004, DATE005, SHIP002"
175,LC-0TPAC5,2026-01-10,2028-06-10,882.0,"AMT002, LEI004, DATE007, SHIP002, SHIP005"
187,LC-PXNF7C,2026-06-01,2026-03-01,-92.0,"DATE004, DATE005"


## Day 62 — Missing Data Strategies.

Part 1: Where is data missing in YOUR dataset?

In [12]:
#12
# Count nulls per column, only show columns that HAVE nulls
missing = report.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

additional_conditions    175
confirming_bank_name     148
confirming_bank_swift    148
error_codes               72
error_messages            72
port_of_loading           23
issue_date                 4
validity_days              4
issuing_bank_swift         4
lc_number                  2
beneficiary_lei            2
applicant_address          2
applicant_name             1
applicant_lei              1
port_of_discharge          1
dtype: int64

Part 2: The NaN Propagation Trap -This is the dangerous part. NaN is contagious — any math involving NaN gives NaN:

In [13]:
# 13
import numpy as np

# NaN infects everything it touches
print(f"1 + NaN = {1 + np.nan}")  # NaN propagation.
print(f"NaN == NaN: {np.nan == np.nan}")  # NaN is not equal to itself  if pd.isna(row['issue_date']):   # ✓
#   if row['issue_date'] == np.nan:  # NEVER True, even when it IS NaN
# This is exactly why your validators use the two-guard pattern — if not x + if pd.isna(x). The == NaN check would silently fail every time.
print(f"NaN > 5: {np.nan > 5}")  # comparisons with NaN are always False

1 + NaN = nan
NaN == NaN: False
NaN > 5: False


DANGER: Silent filter failure

In [14]:
# 14
# DANGER: Silent filter failure
small_df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'score': [85, np.nan, 92, 78]
})

# Trying to filter "scores below 80" — what happens to Bob?
small_df[small_df['score'] < 80]

,name,score
3,Diana,78.0


The safe way

In [15]:
# 15
# Option A: Explicitly keep NaN in the result
small_df[(small_df['score'] < 80) | small_df['score'].isna()]

# Option B: Explicitly exclude NaN (acknowledge what you're doing)
# small_df[small_df['score'].notna() & (small_df['score'] < 80)]

# Option C: Fill before filtering
# small_df['score'].fillna(0) < 80

,name,score
1,Bob,NaN
3,Diana,78.0


Part 3: The Filling Strategies

In [16]:
# 16
# Create a small demo DataFrame
demo = pd.DataFrame({
    'currency': ['USD', 'EUR', None, 'USD', 'EUR', None, 'USD'],
    'amount': [100.0, np.nan, 150.0, 200.0, np.nan, 250.0, 180.0]
})
demo

,currency,amount
0,USD,100.0
1,EUR,NaN
2,NaN,150.0
3,USD,200.0
4,EUR,NaN
5,NaN,250.0
6,USD,180.0


fill strategy and compare

In [17]:
# 17
# Strategy 1: Fill with a constant
demo_v1 = demo.fillna({'currency': 'UNKNOWN', 'amount': 0})

# Strategy 2: Fill with the column mean (only for numerics)
demo_v2 = demo.copy()
demo_v2['amount'] = demo_v2['amount'].fillna(demo_v2['amount'].mean())

# Strategy 3: Forward fill — use the previous valid value
demo_v3 = demo.ffill()

# Strategy 4: Backward fill — use the next valid value
demo_v4 = demo.bfill()

print("Strategy 1 — Constant fill:")
print(demo_v1)
print("\nStrategy 2 — Mean fill (amount only):")
print(demo_v2)
print("\nStrategy 3 — Forward fill:")
print(demo_v3)
print("\nStrategy 4 — Backward fill:")
print(demo_v4)

Strategy 1 — Constant fill:
  currency  amount
0      USD   100.0
1      EUR     0.0
2  UNKNOWN   150.0
3      USD   200.0
4      EUR     0.0
5  UNKNOWN   250.0
6      USD   180.0

Strategy 2 — Mean fill (amount only):
  currency  amount
0      USD   100.0
1      EUR   176.0
2      NaN   150.0
3      USD   200.0
4      EUR   176.0
5      NaN   250.0
6      USD   180.0

Strategy 3 — Forward fill:
  currency  amount
0      USD   100.0
1      EUR   100.0
2      EUR   150.0
3      USD   200.0
4      EUR   200.0
5      EUR   250.0
6      USD   180.0

Strategy 4 — Backward fill:
  currency  amount
0      USD   100.0
1      EUR   150.0
2      USD   150.0
3      USD   200.0
4      EUR   250.0
5      USD   250.0
6      USD   180.0


Part 4: Strategy Decision Framework

The pro move: add an is_missing flag column before filling. Let's apply this to your data:

In [18]:
# 18
# Instead of destroying information, preserve it
report['had_missing_port'] = report['port_of_loading'].isna()

# Now you can fill if needed, but still know where nulls were
# report['port_of_loading'] = report['port_of_loading'].fillna('UNKNOWN')

# Verify
print(f"Missing ports: {report['had_missing_port'].sum()}")

Missing ports: 23


23 — matches exactly what we saw earlier. Now you have a boolean flag you can use in any analysis: group by it, filter by it, count it. And critically, even if someone later fills port_of_loading, you still know which rows were originally missing.
The principle: never destroy information silently. If you're going to impute, always leave a trail so audits and debugging are possible.

Part 5: The .dropna() Escape Hatch

In [19]:
# 19
# Sometimes you just want to remove NaN rows. But be careful:
# How many rows have ANY missing value?
any_missing = report.dropna() # dropna() by default drops rows with ANY missing value across ANY column.
print(f"Original: {len(report)} rows")
print(f"After dropna(): {len(any_missing)} rows")

Original: 200 rows
After dropna(): 3 rows


In [20]:
#20
# Drop only if CRITICAL columns are missing
critical = report.dropna(subset=['transaction_id','amount', 'currency'])
print(f"Keeping rows with these critical fields: {len(critical)} rows")

# Drop only if ALL values are missing (very rare)
all_null = report.dropna(how="all")
print(f"Drop rows where EVERYTHING is null: {len(all_null)} rows")

# Drop columns that are mostly empty (>50% null)
mostly_empty_cols = report.dropna(thresh=len(report))

# Drop columns that are mostly empty (>50% null)
mostly_empty_cols = report.dropna(thresh=len(report) * 0.5, axis=1)
print(f"Keep only columns with at least 50% data: {mostly_empty_cols.shape[1]} columns (was {report.shape[1]})")




Keeping rows with these critical fields: 200 rows
Drop rows where EVERYTHING is null: 200 rows
Keep only columns with at least 50% data: 48 columns (was 51)


Part 6: Pro Technique — Quantify What's Missing
Before deciding how to handle missing data, professionals always quantify it:

In [21]:
#21
# Full missingness report

missing_report = pd.DataFrame({
    'missing_count': report.isnull().sum(),
    'missing_pct': (report.isnull().sum() / len(report) * 100).round(1),
    'dtype': report.dtypes
}).query("missing_count > 0").sort_values('missing_pct', ascending=False)

missing_report


,missing_count,missing_pct,dtype
additional_conditions,175,87.5,category
confirming_bank_name,148,74.0,category
confirming_bank_swift,148,74.0,category
error_codes,72,36.0,str
error_messages,72,36.0,str
port_of_loading,23,11.5,category
issue_date,4,2.0,datetime64[us]
validity_days,4,2.0,float64
issuing_bank_swift,4,2.0,category
lc_number,2,1.0,str


##  Day 63 — Performance & Vectorization

Part 1: The speed hierarchy

There are four ways to do the same thing in pandas, each with dramatically different speed:

In [22]:
# 22
import time   # for timing the benchmarks later
import numpy as np   # already imported, but re-imported here for clarity

# Let's create a bigger DataFrame for meaningful benchmarks
big_df = pd.DataFrame({
    'amount': np.random.uniform(1000, 100000, size=100_000),  # generates 100,000 random float values between 1,000 and 100,000
    'currency': np.random.choice(['USD', 'EUR', 'GBP', 'JPY'], size=100_000) # randomly picks one of those 4 currencies for each of the 100,000 rows
})
print(f"Rows: {len(big_df):,}")



Rows: 100,000


In [23]:
# 24
# string concatenation:
# SLOW: apply for string building
start = time.time()
slow = big_df.apply(lambda row: f"{row['currency']}_{row['amount']:.0f}", axis=1)
time_slow = time.time() - start

#  _ axis=0 (default) — apply the function to each column                                                                                                        #    axis=1 — apply the function to each row
#  _The lambda builds a "STRING" like "USD_73241" for each row:
#  _  'USD' + '_' + '73241'  →  'USD_73241'
# :.0f means format the float with 0 decimal places
# So here, row is an entire row as a Series


# FAST: vectorized string operations



In [24]:
# 23
# - Method 1: Python for loop (the worst)
start = time.time()  # record the start timestamp
result1 = []   # empty list to collect results
for i in range(len(big_df)):                # i = 0, 1, 2, ... 99999
    if big_df.iloc[i]['amount'] > 50000:  # look up row i, grab 'amount'
        result1.append('high')
    else:
        result1.append('low')
time_loop = time.time() - start   # elapsed seconds = now - start

# -  Method 2: .apply() with lambda
start = time.time()
result2 = big_df['amount'].apply(lambda x: 'high' if x > 50000 else 'low')
#  Instead of looping manually with for i in range(...), you hand the whole column to .apply() and give it a function to run on each value.
# .apply() calls that function once per value in the column — so x is just the amount number each time (e.g. 73241.5, 12034.8, etc.).
time_apply = time.time() - start

# - Method 3: np.where (vectorized)
# _  (the operation is applied to the whole array at once at the C level, not row by row at the Python level.)
start = time.time()
result3 = np.where(big_df['amount'] > 50000, 'high', 'low')  #$   np.where(condition, value_if_true, value_if_false)
# big_df['amount'] > 50000 — evaluates the entire column at once, producing an array of True/False for all 100,000 rows simultaneously
time_numpy = time.time() - start


# -  Method 4: Boolean indexing (vectorized)
start = time.time()
result4 = pd.Series('low', index=big_df.index)
result4[big_df['amount'] > 50000] = 'high'
time_boolean = time.time() - start

# Creates a Series of 100,000 rows, every value set to 'low' by default.
# index=big_df.index just makes sure it has the same row labels (0 to 99,999) as the original DataFrame.

print(f"For loop:    {time_loop:.4f}s")
print(f"Apply:       {time_apply:.4f}s")
print(f"np.where:    {time_numpy:.4f}s")
print(f"Boolean:     {time_boolean:.4f}s")



For loop:    1.1362s
Apply:       0.0092s
np.where:    0.0007s
Boolean:     0.0029s


Part 2: Common anti-patterns and their fixes

In [25]:
# 24
# - SLOW: apply for string building
start = time.time()
slow = big_df.apply(lambda row: f"{row['currency']}_{row['amount']:.0f}", axis=1)
time_slow = time.time() - start

#  _ axis=0 (default) — apply the function to each column
#  _ axis=1 — apply the function to each row

# row is an entire row as a Series — it has both currency and amount. That's why you can do row['currency'] and row['amount'] inside the lambda.
# The lambda builds a string like "USD_73241" for each row:
 # 'USD' + '_' + '73241'  →  'USD_73241'
 # :.0f means format the float with 0 decimal places

# - FAST: vectorized string operations
start = time.time()
fast = big_df['currency'] + '_' + big_df['amount'].astype(int).astype(str)
time_fast = time.time() - start

  # _big_df['amount'].astype(int)   # 73241.8  →  73241  (drop decimals)                                                                                                            .astype(str)   # 73241    →  '73241' (make it a string)

  # _  The + operator on two Series concatenates them element-wise across the whole column at once — no Python loop, no row packing.



print(f"Apply axis=1:  {time_slow:.4f}s")
print(f"Vectorized:    {time_fast:.4f}s")
print(f"Speedup:       {time_slow/time_fast:.0f}x")

"""
  So the hierarchy holds:
  - numeric vectorization (np.where) — fastest
  - string vectorization (column + column) — fast
  - .apply(axis=1) — slow
  - for loop with .iloc — the worst
"""


Apply axis=1:  0.3033s
Vectorized:    0.0147s
Speedup:       21x


'\n  So the hierarchy holds:\n  - numeric vectorization (np.where) — fastest\n  - string vectorization (column + column) — fast\n  - .apply(axis=1) — slow\n  - for loop with .iloc — the worst\n'

 groupby pattern

In [26]:
# 25
# SLOW: Loop through groups manually
start = time.time()
means_slow = {}
for currency in big_df['currency'].unique():  # loops over ['USD', 'EUR', 'GBP', 'JPY']  Gets the 4 unique currencies and iterates over them one by one.
    subset = big_df[big_df['currency'] == currency]  # filter rows for this currency
    means_slow[currency] = subset['amount'].mean()   # compute mean of that subset
time_slow = time.time() - start

"""
  Each iteration:
  1. Scans the entire 100,000-row DataFrame to find rows matching that currency
  2. Extracts them as a new DataFrame (subset)
  3. Computes the mean

  So it scans all 100,000 rows 4 times — once per currency.
"""


# FAST: Built-in groupby
start = time.time()
means_fast = big_df.groupby('currency')['amount'].mean()
time_fast = time.time() - start

"""
  big_df.groupby('currency')   # split the DataFrame into 4 groups internally
        ['amount']             # from each group, grab just the amount column
        .mean()                # compute the mean of each group

  Pandas scans the 100,000 rows once, routing each row into the right bucket (USD, EUR, GBP, JPY) as it goes. When it's done, it computes the mean for each bucket.
  No re-scanning.

  The slow version scanned 100,000 rows 4 times = 400,000 row touches. This scans once = 100,000 row touches.

  The result is a Series (not a dict like before):
  currency
  EUR    50012.3
  GBP    49876.1
  JPY    50541.2
  USD    50234.1
"""

print(f"Manual loop:  {time_slow:.4f}s")
print(f"GroupBy:      {time_fast:.4f}s")
print(f"Speedup:      {time_slow/time_fast:.0f}x")



Manual loop:  0.0061s
GroupBy:      0.0027s
Speedup:      2x


In [27]:
#26
# Let's benchmark all three approaches on 100K rows

# Setup: add error_count to big_df
big_df['error_count'] = np.random.randint(0, 8, size=100_000)

# SLOW: apply with named function
def classify_risk(n):
    if n == 0: return 'no_risk'
    elif n <= 2: return 'low_risk'
    elif n <= 4: return 'medium_risk'
    else: return 'high_risk'

start = time.time()
r1 = big_df['error_count'].apply(classify_risk)
time_apply = time.time() - start

# FAST: np.select
start = time.time()
conditions = [
    big_df['error_count'] == 0,
    big_df['error_count'] <= 2,
    big_df['error_count'] <= 4,
]
choices = ['no_risk', 'low_risk', 'medium_risk']
r2 = np.select(conditions, choices, default='high_risk')
time_select = time.time() - start

# Verify same result
print(f"Apply:      {time_apply:.4f}s")
print(f"np.select:  {time_select:.4f}s")
print(f"Speedup:    {time_apply/time_select:.0f}x")
print(f"Same result: {(r1 == r2).all()}")

"""
  The pattern to remember:
  - 2 outcomes → np.where
  - 3+ outcomes → np.select
"""

Apply:      0.0118s
np.select:  0.0027s
Speedup:    4x
Same result: True


'\n  The pattern to remember:\n  - 2 outcomes → np.where\n  - 3+ outcomes → np.select\n'

## Day 64 is Window Functions & Time

Part 1: Cumulative Sum — Running Totals

In [28]:
#27
# Sort flagged by transaction_id to simulate processing order
flagged_sorted = flagged.sort_values('transaction_id').reset_index(drop=True)

# Cumulative error count — "how many total errors have we seen so far?"
flagged_sorted['cumulative_errors'] = flagged_sorted['error_count'].cumsum()

flagged_sorted[['transaction_id', 'error_count', 'cumulative_errors']].head(10)












,transaction_id,error_count,cumulative_errors
0,LC-0LVSNU,1,1
1,LC-0OXFQ8,2,3
2,LC-0TPAC5,5,8
3,LC-0Y9DOM,1,9
4,LC-15Z8VM,1,10
5,LC-1B0Z3N,1,11
6,LC-1EPKYR,1,12
7,LC-1UDBFW,1,13
8,LC-25SQ8C,1,14
9,LC-2ISQP4,2,16


Part 2: Rolling Windows — Moving Averages

In [29]:
# 28
# Rolling average of error_count over a window of 5 transactions
flagged_sorted['rolling_avg_errors'] = (
    flagged_sorted['error_count']
    .rolling(window=5)   # look at 5 rows at a time
    .mean()              # average those 5
    .round(2)            # round to 2 decimal places
)

"""
 flagged_sorted['rolling_avg_errors'] = (
      flagged_sorted['error_count']
      .rolling(window=5)   # look at 5 rows at a time
      .mean()              # average those 5
      .round(2)            # round to 2 decimal places
  )

  Think of it as a sliding window moving down the rows:

  Row 0:  window = [1]              → not enough yet → NaN
  Row 1:  window = [1, 2]           → not enough yet → NaN
  Row 2:  window = [1, 2, 5]        → not enough yet → NaN
  Row 3:  window = [1, 2, 5, 1]     → not enough yet → NaN
  Row 4:  window = [1, 2, 5, 1, 1]  → (1+2+5+1+1)/5 = 2.0  ✓
  Row 5:  window = [2, 5, 1, 1, 1]  → (2+5+1+1+1)/5 = 2.0  ✓
  Row 6:  window = [5, 1, 1, 1, 1]  → (5+1+1+1+1)/5 = 1.8  ✓
  The first 4 rows get NaN because you can't form a complete window of 5 yet.
  The cumulative sum (cell 27) tells you the total. The rolling average tells you the trend.
  If your rolling average climbs from 1.0 to 3.5 over time, that means
  the quality of transactions being processed is getting worse — something a simple total would hide
"""
flagged_sorted[['transaction_id', 'error_count', 'cumulative_errors', 'rolling_avg_errors']].head(10)

,transaction_id,error_count,cumulative_errors,rolling_avg_errors
0,LC-0LVSNU,1,1,NaN
1,LC-0OXFQ8,2,3,NaN
2,LC-0TPAC5,5,8,NaN
3,LC-0Y9DOM,1,9,NaN
4,LC-15Z8VM,1,10,2.0
5,LC-1B0Z3N,1,11,2.0
6,LC-1EPKYR,1,12,1.8
7,LC-1UDBFW,1,13,1.0
8,LC-25SQ8C,1,14,1.0
9,LC-2ISQP4,2,16,1.2


In [30]:
# 29
flagged_sorted['error_count'].rolling(window=5, min_periods=1).mean().round(2).head(5)

0    1.00
1    1.50
2    2.67
3    2.25
4    2.00
Name: error_count, dtype: float64

Part 3: Diff & Pct_change — Detecting Shifts

"how much did this value change from the previous row?"

In [31]:
# 30
# diff() — absolute change
flagged_sorted['error_diff'] = flagged_sorted['error_count'].diff()

# pct_change() — percentage change
flagged_sorted['error_pct_change'] = (
    flagged_sorted['error_count'].pct_change().round(2)
)

flagged_sorted[['transaction_id', 'error_count', 'error_diff', 'error_pct_change']].head(10)


,transaction_id,error_count,error_diff,error_pct_change
0,LC-0LVSNU,1,NaN,NaN
1,LC-0OXFQ8,2,1.0,1.0
2,LC-0TPAC5,5,3.0,1.5
3,LC-0Y9DOM,1,-4.0,-0.8
4,LC-15Z8VM,1,0.0,0.0
5,LC-1B0Z3N,1,0.0,0.0
6,LC-1EPKYR,1,0.0,0.0
7,LC-1UDBFW,1,0.0,0.0
8,LC-25SQ8C,1,0.0,0.0
9,LC-2ISQP4,2,1.0,1.0


Part 4: Expanding Windows — Cumulative Stats

In [32]:
# 31
# Expanding mean — "what's the average error count UP TO this point?"
flagged_sorted['expanding_avg'] = (
    flagged_sorted['error_count']
    .expanding()
    .mean()
    .round(2)
)

flagged_sorted[['transaction_id', 'error_count', 'rolling_avg_errors', 'expanding_avg']].head(10)

,transaction_id,error_count,rolling_avg_errors,expanding_avg
0,LC-0LVSNU,1,NaN,1.00
1,LC-0OXFQ8,2,NaN,1.50
2,LC-0TPAC5,5,NaN,2.67
3,LC-0Y9DOM,1,NaN,2.25
4,LC-15Z8VM,1,2.0,2.00
5,LC-1B0Z3N,1,2.0,1.83
6,LC-1EPKYR,1,1.8,1.71
7,LC-1UDBFW,1,1.0,1.62
8,LC-25SQ8C,1,1.0,1.56
9,LC-2ISQP4,2,1.2,1.60


Part 5: Window Functions with GroupBy

The real power — windows within groups. What if you want the cumulative error count per currency separately?

In [33]:
#32
# Cumulative errors per currency (each currency starts at 0)
flagged_sorted['cum_errors_by_currency'] = (
    flagged_sorted
    .groupby('currency')['error_count']
    .cumsum()
)

flagged_sorted[['transaction_id', 'currency', 'error_count', 'cum_errors_by_currency']].head(15)

,transaction_id,currency,error_count,cum_errors_by_currency
0,LC-0LVSNU,JPY,1,1
1,LC-0OXFQ8,USD,2,2
2,LC-0TPAC5,GBP,5,5
3,LC-0Y9DOM,GBP,1,6
4,LC-15Z8VM,USD,1,3
5,LC-1B0Z3N,USD,1,4
6,LC-1EPKYR,EUR,1,1
7,LC-1UDBFW,JPY,1,2
8,LC-25SQ8C,EUR,1,2
9,LC-2ISQP4,USD,2,6


In [34]:
# Quick diagnostic in your notebook (or as a debug cell)
df_full = pd.read_csv('../data/output/validation_report.csv')
clean_eur = df_full.query("validation_status == 'CLEAN' and currency == 'EUR'")
print(f"EUR + CLEAN count: {len(clean_eur)}")

EUR + CLEAN count: 17
